In [9]:
%load_ext autoreload
%autoreload 2
import logging
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from bgflow.utils import (assert_numpy, distance_vectors, distances_from_vectors, 
                          remove_mean, IndexBatchIterator, LossReporter, as_numpy, compute_distances
)
from bgflow import (GaussianMCMCSampler, DiffEqFlow, BoltzmannGenerator, Energy, Sampler, 
                    MultiDoubleWellPotential, MeanFreeNormalDistribution, 
                    KernelDynamics,BlackBoxDynamics, TimeIndependentDynamics, BruteForceEstimator, HutchinsonEstimator, DenseNet)

from bgflow.utils.autograd import brute_force_jacobian_trace

from tbg.gcl import E_GCL_vel, E_GCL, GCL
from tbg.models2 import EGNN
from ema_pytorch import EMA

import os
import time
import zuko

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# first define system dimensionality and a target energy/distribution

dim = 8
n_particles = 4
n_dimensions = dim // n_particles

# DW parameters
a=0.9
b=-4
c=0
offset=4

target = MultiDoubleWellPotential(dim, n_particles, a, b, c, offset, two_event_dims=False)

# define a MCMC sampler to sample from the target energy

dw4_data = np.load("../data/dw4-dataidx.npy", allow_pickle=True)
all_data = remove_mean(dw4_data[0], n_particles, n_dimensions)
idx = dw4_data[1]
data = all_data[idx[:100000]]
val_data = all_data[idx[100000:500000]]
data_holdout = all_data[idx[-500000:]]
data_holdout.shape

dists_data = as_numpy(compute_distances(data, n_particles, n_dimensions))

#plt.hist(dists_data.reshape(-1), bins=100);

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
n_data = 100000
data_smaller = data[:n_data].clone()
dists_data = as_numpy(compute_distances(data_smaller, n_particles, n_dimensions))
sample_dir = "sample/2d_nf"
model_dir = "models/2d_nf"
fig_dir = "figs/2d_nf"

os.makedirs(sample_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [3]:
# # now set up a prior
# prior =  MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False).cuda()

# now set up a prior
class MyMeanFreeNormalDistribution(Energy, Sampler):
    """ Mean-free normal distribution. """

    def __init__(self, dim, n_particles, std=1., two_event_dims=True):
        if two_event_dims:
            super().__init__([n_particles, dim // n_particles])
        else:
            super().__init__(dim)
        self._two_event_dims = two_event_dims
        self._dim = dim
        self._n_particles = n_particles
        self._spacial_dims = dim // n_particles
        self.register_buffer("_std", torch.as_tensor(std))

    def forward(self, n_samples, temperature=1.):
        samples = self.sample(n_samples, temperature)
        energy = 0.5 * samples.pow(2).sum(dim=-1, keepdim=True) / self._std ** 2
        return samples, energy.squeeze()
        
    def _energy(self, x):
        x = self._remove_mean(x).view(-1, self._dim)
        # TODO: make consistent
        return 0.5 * x.pow(2).sum(dim=-1, keepdim=True) / self._std ** 2

    def sample(self, n_samples, temperature=1.):
        x = torch.ones((n_samples, self._n_particles, self._spacial_dims), dtype=self._std.dtype,
                         device=self._std.device).normal_(mean=0, std=self._std)
        x = self._remove_mean(x)
        if not self._two_event_dims:
            x = x.view(-1, self._dim)
        return x

    def _remove_mean(self, x):
        x = x.view(-1, self._n_particles, self._spacial_dims)
        x = x - torch.mean(x, dim=1, keepdim=True)
        return x



prior =  MyMeanFreeNormalDistribution(dim, n_particles, two_event_dims=False).cuda()

In [11]:
from tarflow import Model
import math
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

img_size = 8
channel_size = 1

# we use a small model for fast demonstration, increase the model size for better results
patch_size = 1
channels = 128
blocks = 4
layers_per_block = 4

n_epochs=100
dim = 8
flow = Model(in_channels=channel_size, img_size=img_size, patch_size=patch_size, 
              channels=channels, num_blocks=blocks, layers_per_block=layers_per_block).to(device)
ema_flow = Model(in_channels=channel_size, img_size=img_size, patch_size=patch_size, 
              channels=channels, num_blocks=blocks, layers_per_block=layers_per_block).to(device)

ema = EMA(
    flow,
    beta = 0.999,              # exponential moving average factor
    update_after_step = 5_000,    # only after this number of .update() calls will it start updating
    update_every = 1,          # how often to actually update, to save on compute (updates every 10th .update() call)
)

optimizer = torch.optim.AdamW(flow.parameters(), 5e-4, weight_decay=1e-2)

class CosineLRSchedule(torch.nn.Module):
    counter: torch.Tensor

    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float, max_lr: float):
        super().__init__()
        self.register_buffer('counter', torch.zeros(()))
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.optimizer = optimizer
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.set_lr(min_lr)

    def set_lr(self, lr: float) -> float:
        if self.min_lr <= lr <= self.max_lr:
            for pg in self.optimizer.param_groups:
                pg['lr'] = lr
        return max(self.min_lr, min(self.max_lr, lr))

    def step(self) -> float:
        with torch.no_grad():
            counter = self.counter.add_(1).item()
        if self.counter <= self.warmup_steps:
            new_lr = self.min_lr + counter / self.warmup_steps * (self.max_lr - self.min_lr)
            return self.set_lr(new_lr)

        t = (counter - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        new_lr = self.min_lr + 0.5 * (1 + math.cos(math.pi * t)) * (self.max_lr - self.min_lr)
        return self.set_lr(new_lr)


8
8
8
8
8
8
8
8


In [ ]:
import pdb

n_batch = 512
batch_iter = IndexBatchIterator(len(data_smaller), n_batch)
lr_schedule_flow = CosineLRSchedule(optimizer, len(batch_iter), n_epochs * len(batch_iter), 1e-6, 5e-4)

def train():
    i = 0
    for epoch in range(n_epochs):
        for it, idxs in enumerate(batch_iter):
            optimizer.zero_grad()
            x_1 = data_smaller[idxs].cuda()
            z, _, logdets = flow(x_1)
            loss = flow.get_loss(z, logdets)
            if torch.isnan(loss) or torch.isinf(loss):
                print("Loss is nan, skipping")
                loss *= 0.
            loss.backward()
            torch.nn.utils.clip_grad_norm_(flow.parameters(), 1.0)
            torch.nn.utils.clip_grad_value_(flow.parameters(), 1.0)
            optimizer.step()
            lr_schedule_flow.step()
            ema.update()
            i += 1
        if (epoch + 1) % 2 == 0:
            ema_state_dict = {k.replace("ema_model.transform.", "transform.").replace("ema_model.base.", "base."): v for k, v in ema.state_dict().items()}
            ema_flow.load_state_dict(ema_state_dict, strict=False)
            z, _, logdets = ema_flow(x_1)
            ema_loss = ema_flow.get_loss(z, logdets)
            print(f"Step [{epoch+1}/{n_epochs}] - Loss: {loss.item():.4f} - EMA Loss: {ema_loss.item():.4f}")

ema_flow_path = model_dir + f'/ema_flow_tar_v4.pth'
flow_path = model_dir + f'/flow_tar_v4.pth'
train()
# if not os.path.exists(flow_path): # if no saved model, train
#     train()
#     torch.save(flow.state_dict(), flow_path)
#     logging.info(f"Model saved to {flow_path}")
#     torch.save(ema_flow.state_dict(), ema_flow_path)
#     logging.info(f"Model saved to {ema_flow_path}")
# else:
#     # load saved model
#     flow.load_state_dict(torch.load(flow_path))
#     logging.info(f"Model loaded from {flow_path}")
#     ema_flow.load_state_dict(torch.load(ema_flow_path))
#     logging.info(f"Model loaded from {ema_flow_path}")


Step [2/100] - Loss: 0.8107 - EMA Loss: 1.6310
Step [4/100] - Loss: 0.7330 - EMA Loss: 1.6063


In [ ]:
n_samples = 10_00
x0_list, x1_list, log_probs_list = [], [], []

for i in range(100):
    with torch.no_grad():
        x0 = torch.randn(n_samples, 8).cuda()
        x1 = flow.reverse(x0)
        z, _, logdets = flow(x1)
        log_probs = (-0.5 * z.pow(2)).mean(dim=-1) + logdets
        x0_list.append(x0)
        x1_list.append(x1)
        log_probs_list.append(log_probs)

x0 = torch.cat(x0_list)
x1 = torch.cat(x1_list)
log_probs = torch.cat(log_probs_list)
    
print(log_probs.shape)
energies_data = target.energy(data).detach().cpu().numpy()
#energies_compare = sampls.energy()
energies_bg = target.energy(x1).detach().cpu().numpy().flatten()
print(energies_bg.shape)
log_weights = -energies_bg - log_probs.detach().cpu().numpy()
#log_weights = bg.log_weights_given_latent(x1, x0, dlogp1.reshape(-1,1), normalize=False).detach().cpu().numpy()
log_w_np = as_numpy(log_weights)
energies_prior = target.energy(data).detach().cpu().numpy()
print(np.exp(log_w_np).shape)
min_energy = min(energies_data.min(), energies_bg.min(), energies_prior.min())
max_energy = max(energies_data.max(), energies_bg.max(), energies_prior.max())

plt.figure(figsize=(13,8))
efac = 1
#plt.hist(energies_prior, bins=100, density=True, range=(min_energy, 0), alpha=0.4, color="b", histtype='step', linewidth=4,
#         label="prior");
plt.hist(energies_data, bins=100, density=True, range=(min_energy, 0), alpha=0.4, color="g", histtype='step', linewidth=4, 
         label="Target");
plt.hist(energies_bg, bins=100, density=True,  range=(min_energy, 0), alpha=0.4, histtype='step', linewidth=4,
         color="r", label=f"BG samples");

plt.hist(energies_bg.reshape(-1), bins=100, density=True,  range=(min_energy, 0), alpha=0.4, histtype='step', linewidth=4,
         color="b", label=f"Weighted BG samples", weights=np.exp(log_w_np));
#plt.hist(energies_compare, bins=100, density=True, range=(energies_bg.min(), 0), alpha=0.4, color="y", 
#         label="MCMC data (test)");
plt.title("DW4 Energy distribution", fontsize=45)
plt.xlabel("u(x)", fontsize=45)  
plt.xticks(fontsize=45) 
plt.yticks(fontsize=45);
plt.legend(fontsize=25);

In [7]:
energies_bg

array([-10.998799, -15.534412, -17.849583, ..., -17.035902,  41.21791 ,
       -16.105434], dtype=float32)

In [8]:
energies_data

array([[-20.032698],
       [-22.73768 ],
       [-18.32373 ],
       ...,
       [-23.264725],
       [-19.408096],
       [-22.277653]], dtype=float32)